In [1]:
IMAGE_PATH_BASE = 'AllImages'
INPUT_FILE_PATH = './FinalDataset.csv'

In [ ]:
# Export FinalDataset to csv

import pandas as pd
import os
from PIL import Image
from tqdm.notebook import tqdm
import numpy as np
from PIL import Image
import os
from glob import glob
from tqdm.notebook import tqdm

def downscale(image : Image, size):
    image.thumbnail((size, size), Image.ANTIALIAS)
    return image

def crop(image : Image, size : int):
    width, height = image.size
    
    left = (width - size) / 2
    top = (height - size) / 2
    right = (width + size) / 2
    bottom = (height + size) / 2
    
    return image.crop((left, top, right, bottom))

result_dataset_path = f"{IMAGE_PATH_BASE}"
resized_images_base_path = "../Step 2 - Image dataset/Images"

if not os.path.exists(result_dataset_path):
    os.mkdir(result_dataset_path) 

dataset = pd.read_csv(f"{INPUT_FILE_PATH}", dtype={
    "Id": "string",
    "GZ12_Id": "string",
    "GZH_Id": "string",
    "GZC_Id": "string",
    "GZD_Id": "string",
    "Class": "string",
    'OriginalDatasetId': "string"})

gzh_dataset = pd.read_csv("GZH/gz_hubble_main.csv")

empty_galaxy_count = 0
for _, row in tqdm(dataset.iterrows(), total = len(dataset)):
    written = False

    if not os.path.exists(f"{result_dataset_path}/{row['Id']}"):
        os.mkdir(f"{result_dataset_path}/{row['Id']}")

    if not pd.isna(row["GZ12_Id"]):
        if os.path.exists(f"{resized_images_base_path}/GZ12_All_Resized/{row['GZ12_Id']}.jpg"):
            gz12_img = Image.open(f"{resized_images_base_path}/GZ12_All_Resized/{row['GZ12_Id']}.jpg")
            gz12_img.save(f"{result_dataset_path}/{row['Id']}/{row['GZ12_Id']}.jpg")
            written = True

    if not pd.isna(row["GZH_Id"]):
        gzh_id = gzh_dataset[gzh_dataset["zooniverse_id"] == row["GZH_Id"]]["survey_id"].iloc[0]
        if os.path.exists(f"{resized_images_base_path}/GZH_Resized/{gzh_id}.jpg"):
            gzh_img = Image.open(f"{resized_images_base_path}/GZH_Resized/{gzh_id}.jpg")
            gzh_img.save(f"{result_dataset_path}/{row['Id']}/{row['GZH_Id']}.jpg") # save as original id to match db naming
            written = True

    if not pd.isna(row["GZC_Id"]) and os.path.exists(f"{resized_images_base_path}/GZC_Resized/{row['GZC_Id']}.jpg"):
        gzc_img = Image.open(f"{resized_images_base_path}/GZC_Resized/{row['GZC_Id']}.jpg")
        gzc_img.save(f"{result_dataset_path}/{row['Id']}/{row['GZC_Id']}.jpg")
        written = True

    if not pd.isna(row["GZD_Id"]) and os.path.exists(f"{resized_images_base_path}/GZD_Resized/{row['GZD_Id']}.jpg"):
        gzd_img = Image.open(f"{resized_images_base_path}/GZD_Resized/{row['GZD_Id']}.jpg")
        gzd_img.save(f"{result_dataset_path}/{row['Id']}/{row['GZD_Id']}.jpg")
        written = True

    if not written:
        empty_galaxy_count = empty_galaxy_count + 1

print(empty_galaxy_count)

In [2]:
import tensorflow as tf

IMAGE_SIZE = 128 #@param
CLASSES = [b"Elliptical", b"Spiral"] # 2 classes
AUTO = tf.data.experimental.AUTOTUNE

def decode_jpeg_and_label(item):
  filename = item['image_path']
  galaxy_class = item['class']
  objid = item['objid']
  ids = item['id']
  
  bits = tf.io.read_file(filename)
  image = tf.image.decode_jpeg(bits)
  return image, galaxy_class, objid, ids

def recompress_image(image, label, objid, ids):
  image = tf.image.encode_jpeg(image, optimize_size=True, chroma_downsampling=False)
  return image, label, objid, ids

def _bytestring_feature(list_of_bytestrings):
  return tf.train.Feature(bytes_list=tf.train.BytesList(value=list_of_bytestrings))

def _int_feature(list_of_ints): # int64
  return tf.train.Feature(int64_list=tf.train.Int64List(value=list_of_ints))

def _float_feature(list_of_floats): # float32
  return tf.train.Feature(float_list=tf.train.FloatList(value=list_of_floats))

def to_tfrecord(tfrec_filewriter, img_bytes, label, objid, id):  
  class_num = np.argmax(np.array(CLASSES)==label) # 'roses' => 2 (order defined in CLASSES)
  one_hot_class = np.eye(len(CLASSES))[class_num]     # [0, 0, 1, 0, 0] for class #2, roses

  feature = {
      "image": _bytestring_feature([img_bytes]), # one image in the list
      "class": _int_feature([class_num]),        # one class in the list
      "label":         _bytestring_feature([label]),          # fixed length (1) list of strings, the text label
      "objid":         _bytestring_feature([objid]),          # fixed length (1) list of strings, the text label
      "id":         _bytestring_feature([objid]),          # fixed length (1) list of strings, the text label
      "one_hot_class": _float_feature(one_hot_class.tolist()) # variable length  list of floats, n=len(CLASSES)
  }
  return tf.train.Example(features=tf.train.Features(feature=feature))

def create_tf_record_dataset(filenames, items_per_record, output_folder):
  dataset2 = filenames.map(decode_jpeg_and_label, num_parallel_calls=AUTO)
  dataset3 = dataset2.map(recompress_image, num_parallel_calls=AUTO)
  dataset3 = dataset3.batch(items_per_record) # sharding: there will be one "batch" of images per file

  # print("Writing TFRecords")
  for shard, (image, label, ids, objid) in enumerate(dataset3):
    # batch size used as shard size here
    shard_size = image.numpy().shape[0]
    filename = output_folder + "/{:02d}-{}.tfrec".format(shard, shard_size)
    
    np_image = image.numpy()
    np_label = label.numpy()
    np_objid = objid.numpy()
    np_id = ids.numpy()

    with tf.io.TFRecordWriter(filename) as out_file:
      for i in range(shard_size):
        example = to_tfrecord(out_file,
                              np_image[i], # re-compressed image: already a byte string
                              np_label[i],
                              np_objid[i],
                              np_id)
        out_file.write(example.SerializeToString())
      # print("Wrote file {} containing {} records".format(filename, shard_size))
  # print("Done.")

In [4]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import tensorflow as tf
import pandas as pd
import os

ids_to_path = lambda ids: np.array([f"{IMAGE_PATH_BASE}/{row[0]}/{row[1]}.jpg" for row in ids])

def convert_ids_and_classes_to_dataset(ids, classes):
    dataset_dictionary = {
        "image_path": ids_to_path(ids),
        "class": classes.T,
        "objid": ids[:, 0].T.astype(str),
        "id": ids[:, 1].T.astype(str),
    }

    return tf.data.Dataset.from_tensor_slices(dataset_dictionary)

def image_exists(row, supported_datasets):
    for dataset in supported_datasets:
        if os.path.exists(f"{IMAGE_PATH_BASE}/{row['Id']}/{row[f'{dataset}_Id']}.jpg"):
            return True
    
    return False

def get_flat_image_list(dataset, supported_datasets):
    ids = []
    for index, row in dataset.iterrows():
        for supported_dataset in supported_datasets:
            if not pd.isna(row[f'{supported_dataset}_Id']) and os.path.exists(f"{IMAGE_PATH_BASE}/{row['Id']}/{row[f'{supported_dataset}_Id']}.jpg"):
                ids.append({
                    "Id": row["Id"],
                    "DatasetId": row[f"{supported_dataset}_Id"],
                    "Class": row["BinaryClass"]
                    })
                
                # break # include only one image per galaxy
            
    return pd.DataFrame(ids)


def create_records_for_training_and_test_images(training_data, test_data, supported_datasets, path):

    if not os.path.exists(f"{path}/train"):
        os.makedirs(f"{path}/train") 
        os.makedirs(f"{path}/test") 

    flat_training_images = get_flat_image_list(training_data, supported_datasets)
    flat_test_images = get_flat_image_list(test_data, supported_datasets)
    print("Total training images:", len(flat_training_images))
    print("Total test images:", len(flat_test_images))


    # number_of_galaxies_to_pad = BATCH_SIZE - len(flat_training_images) % BATCH_SIZE

    # resample_indexes = np.random.choice(
    #     np.arange(len(flat_training_images)),
    #     number_of_galaxies_to_pad,
    #     replace=False)

    train_ids = flat_training_images[[f"Id", "DatasetId"]]
    train_classes = flat_training_images[f"Class"]

    # padded_train_ids = np.concatenate((original_train_ids, original_train_ids.iloc[resample_indexes]))
    # padded_train_classes = np.concatenate((original_train_classes, original_train_classes.iloc[resample_indexes]))

    test_ids = flat_test_images[["Id", "DatasetId"]]
    test_classes = flat_test_images["Class"]

    tf_dataset = convert_ids_and_classes_to_dataset(train_ids.to_numpy(), train_classes.to_numpy())
    create_tf_record_dataset(tf_dataset, 4096, f'{path}/train')

    tf_dataset = convert_ids_and_classes_to_dataset(test_ids.to_numpy(), test_classes.to_numpy())
    create_tf_record_dataset(tf_dataset, 4096, f'{path}/test')



BATCH_SIZE = 1024
SEED = 17
N_FOLDS = 10

np.random.seed(SEED)

for bin, supported_datasets in [
    ("Bin_SDSS_z_0.0-0.25", ["GZ12"]),
    ("Bin_DES_z_0.0-0.15", ["GZD"]), 
    ('Bin_Hubble_z_0.0-0.3', ["GZH", "GZC"]),
    ('Bin_Hubble_z_0.3-1.0', ["GZH", "GZC"]),
    ('Bin_Hubble_z_1.0_4.0', ["GZH", "GZC"])]:

    print(bin)

    dataset = pd.read_csv(f"{bin}.csv", dtype={
        "GZ12_Id": "string",
        "GZH_Id": "string",
        "GZC_Id": "string",
        "GZD_Id": "string",
        'OriginalDatasetId': "string"})

    dataset["ImageExists"] = dataset.apply(image_exists, args = [supported_datasets], axis= 1)
    dataset = dataset[dataset["ImageExists"]]

    

    tfdataset_path = f"TFDataset/{bin}"
    if not os.path.exists(tfdataset_path):
        os.makedirs(tfdataset_path) 


    # -------------------------- Tuning data preparation -----------------------------------

    # Use K-Fold to ensure that training and test splits have original class distribution
    # kfold = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)

    # tuning_galaxies = dataset.sample(n = 10_000, replace=False, random_state= SEED) if len(dataset) >= 10_000 else dataset

    # fold = 1
    # for train_index, test_index in kfold.split(tuning_galaxies[["Id", "GZ12_Id", "GZC_Id", "GZD_Id", "GZH_Id"]], tuning_galaxies["BinaryClass"] ):

    #     dataset_folder_base = f"{tfdataset_path}/tuning"

    #     training_data = tuning_galaxies.iloc[train_index]
    #     print("Training Spiral:", len(training_data[training_data["BinaryClass"] == "Spiral"]))
    #     print("Training Elliptical:", len(training_data[training_data["BinaryClass"] == "Elliptical"]))
    #     test_data = tuning_galaxies.iloc[test_index]
    #     print("Test Spiral:", len(test_data[test_data["BinaryClass"] == "Spiral"]))
    #     print("Test Elliptical:", len(test_data[test_data["BinaryClass"] == "Elliptical"]))

    #     create_records_for_training_and_test_images(training_data, test_data, supported_datasets, dataset_folder_base)

    #     break # only 1 fold for tuning

    # print()

    # -------------------------- Ablation data preparation -----------------------------------
    
    # Use K-Fold to ensure that training and test splits have original class distribution
    # kfold = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)

    # ablation_galaxies = dataset.sample(n = 20_000, replace=False, random_state= SEED) if len(dataset) >= 20_000 else dataset

    # fold = 1
    # for train_index, test_index in kfold.split(ablation_galaxies[["Id", "GZ12_Id", "GZC_Id", "GZD_Id", "GZH_Id"]], ablation_galaxies["BinaryClass"] ):

    #     dataset_folder_base = f"{tfdataset_path}/ablation/fold_{fold}"

    #     training_data = ablation_galaxies.iloc[train_index]
    #     print("Training Spiral:", len(training_data[training_data["BinaryClass"] == "Spiral"]))
    #     print("Training Elliptical:", len(training_data[training_data["BinaryClass"] == "Elliptical"]))
    #     test_data = ablation_galaxies.iloc[test_index]
    #     print("Test Spiral:", len(test_data[test_data["BinaryClass"] == "Spiral"]))
    #     print("Test Elliptical:", len(test_data[test_data["BinaryClass"] == "Elliptical"]))

    #     create_records_for_training_and_test_images(training_data, test_data, supported_datasets, dataset_folder_base)

    #     fold += 1
    #     print()

    # -------------------------- Training K-fold data preparation -----------------------------------
    # kfold = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)
    # fold = 1
    # for train_index, test_index in kfold.split(dataset[["Id", "GZ12_Id", "GZC_Id", "GZD_Id", "GZH_Id"]], dataset["BinaryClass"] ):
    #     print("Fold", fold)
    #     dataset_folder_base = f"{tfdataset_path}/training/fold_{fold}"

    #     training_data = dataset.iloc[train_index]
    #     print("Training Spiral:", len(training_data[training_data["BinaryClass"] == "Spiral"]))
    #     print("Training Elliptical:", len(training_data[training_data["BinaryClass"] == "Elliptical"]))
    #     test_data = dataset.iloc[test_index]
    #     print("Test Spiral:", len(test_data[test_data["BinaryClass"] == "Spiral"]))
    #     print("Test Elliptical:", len(test_data[test_data["BinaryClass"] == "Elliptical"]))

    #     create_records_for_training_and_test_images(training_data, test_data, supported_datasets, dataset_folder_base)
    #     fold += 1
    #     print()


    # -------------------------- All data preparation -----------------------------------
    
    # Use K-Fold to ensure that training and test splits have original class distribution
    kfold = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)

    fold = 1
    for train_index, test_index in kfold.split(dataset[["Id", "GZ12_Id", "GZC_Id", "GZD_Id", "GZH_Id"]], dataset["BinaryClass"] ):

        dataset_folder_base = f"{tfdataset_path}/all/fold_{fold}"

        training_data = dataset.iloc[train_index]
        print("Training Spiral:", len(training_data[training_data["BinaryClass"] == "Spiral"]))
        print("Training Elliptical:", len(training_data[training_data["BinaryClass"] == "Elliptical"]))
        test_data = dataset.iloc[test_index]
        print("Test Spiral:", len(test_data[test_data["BinaryClass"] == "Spiral"]))
        print("Test Elliptical:", len(test_data[test_data["BinaryClass"] == "Elliptical"]))

        create_records_for_training_and_test_images(training_data, test_data, supported_datasets, dataset_folder_base)

        fold += 1
        print()

Bin_SDSS_z_0.0-0.25
Training Spiral: 197340
Training Elliptical: 75243
Test Spiral: 21926
Test Elliptical: 8361
Total training images: 272583
Total test images: 30287

Training Spiral: 197340
Training Elliptical: 75243
Test Spiral: 21926
Test Elliptical: 8361
Total training images: 272583
Total test images: 30287

Training Spiral: 197340
Training Elliptical: 75243
Test Spiral: 21926
Test Elliptical: 8361
Total training images: 272583
Total test images: 30287

Training Spiral: 197340
Training Elliptical: 75243
Test Spiral: 21926
Test Elliptical: 8361
Total training images: 272583
Total test images: 30287

Training Spiral: 197339
Training Elliptical: 75244
Test Spiral: 21927
Test Elliptical: 8360
Total training images: 272583
Total test images: 30287

Training Spiral: 197339
Training Elliptical: 75244
Test Spiral: 21927
Test Elliptical: 8360
Total training images: 272583
Total test images: 30287

Training Spiral: 197339
Training Elliptical: 75244
Test Spiral: 21927
Test Elliptical: 8360
